# Bayesian neural networks

In this demo, we gain hands-on experience with fitting Bayesian neural networks (BNNs) using [`numpyro`](https://num.pyro.ai/en/latest/index.html#$0).

The main concepts covered are:
- constructing Bayesian neural networks in numpyro.
- comparing inference algorithms, namely maximum aposteriori (MAP), variational inference (VI), and Markov chain Monte Carlo (MCMC).
- understanding the influence of BNN choices, such as the activation function, width, depth, and priors.

**Outline:**

1.   [Setup](#setup)
2.   [BNN - one layer, narrow width, odd activation](#bnn)
3.   [BNN - one layer, large width, odd activation](#bnn)
4.   [Exercises](#ex)

# Setup <a id='setup'></a>

Let's start by loading the packages needed.

In [ ]:
import os
import time

import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd

#!pip install jax
import jax
import jax.numpy as jnp
import jax.random as random

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

#!pip install numpyro
import numpyro
import numpyro.optim as optim
from numpyro import deterministic

import numpyro.distributions as dist
from numpyro.infer import MCMC, NUTS, SVI, Predictive, Trace_ELBO
from numpyro.infer.autoguide import AutoNormal, AutoMultivariateNormal, AutoDelta


## Motorcycle data

For illustration, we will work with the motorcyle data, which provides a series of measurements of head acceleration in a simulated motorcycle accident, used to test crash helmets.

This is a small data set with only 1 feature (time) and 1 numerical outcome (head acceleration) recorded at $N=133$ time points. Although modern applications involves lots of data, both in terms of the number of observations and number of features, this small data set is useful to visualize and understand the role of various choices in BNNs.

In [ ]:
# Load motorcycle data
mcycle = pd.read_csv('mcycle.csv', index_col=0)
N = mcycle.shape[0]
D = mcycle.shape[1]-1
y = mcycle['accel']
X = mcycle.drop('accel', axis=1)
print('Number of observations: ', N)

# Split into training and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=26)

# Scale the inputs
from sklearn.preprocessing import StandardScaler
ss = StandardScaler()
X_train = jnp.array(ss.fit_transform(X_train))
X_test = jnp.array(ss.transform(X_test))
ss_y = StandardScaler()
y_train = jnp.array(ss_y.fit_transform(y_train.values.reshape(-1, 1)))
y_test = jnp.array(ss_y.transform(y_test.values.reshape(-1, 1)))

# Create a grid of points for prediction
X_grid = np.linspace(X.min(), X.max(), 100).reshape(-1, 1)
X_grid = jnp.array(ss.transform(X_grid)) 

Note: following standard practice, we have scaled the inputs and outputs. 

Let's visualize the data. Notice that data exhibits nonlinearity and heteroskedasticity.

In [ ]:
# Plot the data
sns.scatterplot(mcycle, x = 'times', y = 'accel')
plt.xlabel('time')
plt.ylabel('accelation')
plt.show()

# BNN: one layer, narrow width, odd activation <a id='bnn'></a>

Let's build a simple BNN, with code adapted from [numpyro BNN example](https://num.pyro.ai/en/latest/examples/bnn.html).

We assume a fully-connected, feed-forward architecture with independent normal priors on the weights and biases and only one hidden layer of width $D_H$. 
The biases have standard normal priors: 
$$ b_{1,d} \sim \text{N}(0,1),$$
whereas the variance of the weights scales inversely with the width of the previous layer, e.g. for the final layer:
$$ w_{2,d} \sim \text{N}(0,1/D_H).$$
This scaling is commonly used in BNNs and is crucial to convergence to Gaussian processes in the infinite-width regime. 

We assume  a Gaussian likelihood for the data, 
$$ y_n \sim \text{N}(f_\theta(x_n), 1/\tau),$$
where $f_\theta(x)$ is the final output of our neural network and $\theta$ collects all the weights and biases. Given the evident heteroskedasticity of the model, we may want to consider a more flexible model that also allows the variance to depend on $x$. However, for simplicity, we focus on the standard regression setting, and leave this **extension as an exercise**. Moreover, this also gives us some understanding of the behavior of the model under misspecification. 

The precision has Gamma prior,
$$ \tau \sim \text{Gam}(100,1).$$
Why did we choose the parameters $100$ and $1$? The centers the prior around large values of the precision, and hence small values of observation noise. The choice of these parameters can have a large impact: as an **exercise, trying changing these values**.

In [ ]:
# Activation function we use in our neural network
def acti_fun(x):
    return jnp.tanh(x)

# Define the model
def model(X, Y, D_H, D_Y=1):
    N, D_0 = X.shape
    x = deterministic("x", X) 

    # First layer of the neural network
    # Bias term for the first layer
    b1 = numpyro.sample("b1", dist.Normal(jnp.zeros((D_H,)), jnp.ones((D_H,))))
    assert b1.shape == (D_H,)
    # Weights for the first layer
    w1 = numpyro.sample("w1", dist.Normal(jnp.zeros((D_0, D_H)), jnp.ones((D_0, D_H))/ jnp.sqrt(D_0)))
    assert w1.shape == (D_0, D_H)
    # Activations for the first layer
    z1 = deterministic("z1", acti_fun(x @ w1 + b1)) 
    assert z1.shape == (N, D_H)

    # Final layer of the neural network
    # Bias term for the final layer
    b2 = numpyro.sample("b2", dist.Normal(jnp.zeros((D_Y,)), jnp.ones((D_Y,))))
    assert b2.shape == (D_Y,)
    # Weights for the second layer
    w2 = numpyro.sample("w2", dist.Normal(jnp.zeros((D_H, D_Y)), jnp.ones((D_H, D_Y))/ jnp.sqrt(D_H)))
    assert w2.shape == (D_H, D_Y)
    # Activations for the second layer
    z2 = deterministic("z2", z1 @ w2 + b2)
    assert z2.shape == (N, D_Y)

    if Y is not None:
        assert z2.shape == Y.shape

    # we put a prior on the observation noise
    prec_obs = numpyro.sample("tau", dist.Gamma(100.0, 1.0))
    sigma_obs = 1.0 / jnp.sqrt(prec_obs)

    # observe data
    with numpyro.plate("N", N):
        # note we use to_event(1) because each observation has shape (1,)
        numpyro.sample("Y", dist.Normal(z2, sigma_obs).to_event(1), obs=Y)

In [ ]:
# Render and visualize the model
D_H = 10
D_Y = 1
numpyro.render_model(model, model_args=(X_train, y_train, D_H, D_Y),
    render_distributions=True,
    render_params=True,)

## Inference

Now let's fit the model. We will compare three different inference schemes, Markov chain Monte Carlo (MCMC), stochastic variational inference (SVI), and maximum aposteriori inference (MAP).

### MCMC

Let's start with MCMC. Here we will use the NUTS (no-U-turn sampler). For other MCMC algorithms and options, see https://num.pyro.ai/en/stable/mcmc.html. 

More details for the NUTS algorithm are available here: https://num.pyro.ai/en/latest/mcmc.html#numpyro.infer.hmc.NUTS. For example, the default initialization for all parameters is random initialization from a uniform distribution `init_strategy=<function init_to_uniform>`, and the documentation suggests that this may not be good for some models (see also [alternative initializations](https://num.pyro.ai/en/0.5.0/utilities.html#init-strategy))


In [ ]:
numpyro.set_platform('cpu')
rng_key, rng_key_predict = random.split(random.PRNGKey(0))

In [ ]:
# Define a function to run MCMC
def run_mcmc(model, num_chains, num_samples, num_warmup, rng_key, X, Y, D_H, D_Y):
    start = time.time()
    kernel = NUTS(model)
    mcmc = MCMC(
        kernel,
        num_warmup=num_warmup,
        num_samples=num_samples,
        num_chains=num_chains,
        progress_bar=False if "NUMPYRO_SPHINXBUILD" in os.environ else True,
    )
    mcmc.run(rng_key = rng_key, X=X, Y=Y, D_H = D_H, D_Y = D_Y)
    #mcmc.print_summary()
    print("\nMCMC  time:", time.time() - start)
    return mcmc.get_samples()

In [ ]:
# Run MCMC
num_warmup=2000
num_samples=5000
num_chains=1
numpyro.set_host_device_count(num_chains)

posterior_samples = run_mcmc(model = model,  num_chains = num_chains,num_samples = num_samples, num_warmup = num_warmup, rng_key = rng_key, 
                             X = X_train, Y = y_train, D_H = D_H, D_Y = D_Y)

The output is a dictionary, containing the posterior samples for each variable of the model.

In [ ]:
# Print keys of the posterior samples
print(posterior_samples.keys())
# Print the shape of the posterior samples of z2
print(posterior_samples['z2'].shape)

Let's plot some traceplots to investigate convergence.

In [ ]:
# Traceplots
fig, axs = plt.subplots(1, 3, figsize=(12, 4))
axs = axs.flatten()
# Traceplot for tau
axs[0].plot(posterior_samples['tau'])
axs[0].set_title('Traceplot for tau')
axs[0].set_xlabel('iteration')
axs[0].set_ylabel('tau')
# Traceplot for z2
axs[1].plot(posterior_samples['z2'][:,0,0])
axs[1].set_title('Traceplot of the mean for x=' + str(np.round(X_train[0,0], 2)))
axs[1].set_xlabel('iteration')
axs[1].set_ylabel('mean')
# Traceplot for z2
axs[2].plot(posterior_samples['z2'][:,1,0])
axs[2].set_title('Traceplot of the mean for x=' + str(np.round(X_train[1,0], 2)))
axs[2].set_xlabel('iteration')
axs[2].set_ylabel('mean')


The traceplots do not show problematic signs, so let's proceed to predictive inference. Remember with BNNs, the weights and biases themselves are not identifiable and their posterior is not of direct interest. Instead, we are interested in the posterior predictive distribution over the function $f_\theta(x)$ and the outcomes $y$. For this, we need to draw predictive samples.

In [ ]:
# Draw predictive samples from the posterior predictive distribution
predictive = Predictive(model = model, posterior_samples=posterior_samples)
predictions = predictive(rng_key = rng_key_predict, X=X_test, Y = None, D_H = D_H, D_Y = D_Y)
predictions_grid = predictive(rng_key = rng_key_predict, X=X_grid, Y = None, D_H = D_H, D_Y = D_Y)

Again, the output is dictionary.

In [ ]:
# Print keys of the predictions
print(predictions.keys())
# Print the shape of the predictions of z2
print(predictions['z2'].shape)

Now, we will save some summaries of the predictive samples.

In [ ]:
# Extract the mean and quantiles for the predicted output on the test data
mean_test_mcmc = jnp.mean(predictions['Y'], axis = 0).reshape(-1,)
lower_test_mcmc = jnp.quantile(predictions['Y'], 0.025, axis = 0).reshape(-1,)
upper_test_mcmc = jnp.quantile(predictions['Y'], 0.975, axis = 0).reshape(-1,)

# Extract the mean and quantiles for the predicted function on the grid data
mean_grid_mcmc = jnp.mean(predictions_grid['z2'], axis = 0).reshape(-1,)
lower_grid_mcmc = jnp.quantile(predictions_grid['z2'], 0.025, axis = 0).reshape(-1,)
upper_grid_mcmc = jnp.quantile(predictions_grid['z2'], 0.975, axis = 0).reshape(-1,)

Before visualizing the results, let's run alternative inference strategies and plot the predictions side-by-side for easier comparison.

### SVI

For this example, MCMC didn't take too long to run, but for larger datasets and models it may become infeasible. To speed up inference, we will next use SVI to approximate the posterior. 

Note that the variational posterior is called `guide` in `numpyro`. For details, see http://pyro.ai/examples/svi_part_i.html. 

Here we are assuming a (diagonal) normal using `AutoNormal`, that is, a mean-field assumption where the variational posterior for all weights and bias is assumed to be Gaussian. For other automatic guide options, see https://num.pyro.ai/en/stable/autoguide.html. 

We are using default initialization of the variational parameters (in this case, the means and variances of the Gaussian distributions). For the `AutoNormal`, the locations are randomly initialized from a uniform distribution (`init_loc_fn=<function init_to_uniform>`) and the scales are set to 0.1 (`init_scale=0.1`). Again, this can be changed to other [initialization strategies](https://num.pyro.ai/en/0.5.0/utilities.html#init-strategy).

Here we are using Adam, but for more details on optimizers, see https://num.pyro.ai/en/0.5.0/optimizers.html.

In [ ]:
# Define a function to perform SVI
def do_SVI(model, rng_key, numsteps, X, Y, D_H, D_Y=1):
    start = time.time()
    guide = AutoNormal(model)
    optimizer = optim.Adam(step_size = 0.01)
    svi = SVI(model, guide, optimizer, Trace_ELBO())
    svi_results = svi.run(rng_key = rng_key, num_steps = numsteps, X=X, Y = Y, D_H = D_H, D_Y = D_Y)
    params = svi_results.params
    losses = svi_results.losses
    print("\nSVI  time:", time.time() - start)
    return losses, params, guide

In [ ]:
# Run SVI
numsteps= 1000
svi_losses, svi_params, svi_guide = do_SVI(model=model, rng_key=rng_key, numsteps=numsteps, X = X_train, Y = y_train, D_H=D_H, D_Y=D_Y)

That did speed things up! By a factor of ~11.

Did we run the algorithm long enough? Let's check by plotting the ELBO across the steps of the algorithm.

In [ ]:
# Plot the ELBO to check convergence
plt.figure(figsize=(6, 4))
sns.lineplot(x=np.arange(numsteps), y=-svi_losses)
plt.xlabel('Step')
plt.ylabel('ELBO')
plt.title('ELBO over steps')

The trace of ELBO does not show problematic signs.

Again, we are not directly interested in the varitional posterior and parameters. So, let's take samples from the posterior predictive and save some summaries.

In [ ]:
# Sample from the posterior predictive distribution
num_samples=2000
posterior_predictive = Predictive(model=model, guide=svi_guide, params=svi_params, num_samples=num_samples)
svi_predictions = posterior_predictive(rng_key = rng_key_predict, X=X_test, Y=None, D_H=D_H, D_Y=D_Y)
svi_grid = posterior_predictive(rng_key = rng_key_predict, X=X_grid, Y=None, D_H=D_H, D_Y=D_Y)

In [ ]:
# Extract the mean and quantiles for the predicted output on the test data
mean_test_svi = jnp.mean(svi_predictions['Y'], axis = 0).reshape(-1,)
lower_test_svi = jnp.quantile(svi_predictions['Y'], 0.025, axis = 0).reshape(-1,)
upper_test_svi = jnp.quantile(svi_predictions['Y'], 0.975, axis = 0).reshape(-1,)

# Extract the mean and quantiles for the predicted function on the grid data
mean_grid_svi = jnp.mean(svi_grid['z2'], axis = 0).reshape(-1,)
lower_grid_svi = jnp.quantile(svi_grid['z2'], 0.025, axis = 0).reshape(-1,)
upper_grid_svi = jnp.quantile(svi_grid['z2'], 0.975, axis = 0).reshape(-1,)

### MAP

Lastly, let's also try MAP estimation by changing the variational posterior to AutoDelta (the variational posterior is dirac delta function).

In [ ]:
# Define a function to perform MAP estimation
def do_MAP(model, rng_key, numsteps, X, Y, D_H, D_Y=1):
    start = time.time()
    guide = AutoDelta(model)
    optimizer = optim.Adam(0.01)
    svi = SVI(model, guide, optimizer, Trace_ELBO())
    svi_results = svi.run(rng_key = rng_key, num_steps = numsteps, X=X, Y = Y, D_H = D_H, D_Y = D_Y)
    params = svi_results.params
    losses = svi_results.losses
    print("\nMAP  time:", time.time() - start)
    return losses, params, guide

In [ ]:
# Run MAP estimation
numsteps= 1000
map_losses, map_params, map_guide = do_MAP(model=model, rng_key=rng_key, numsteps=numsteps, X = X_train, Y = y_train, D_H = D_H, D_Y = 1)

That was even faster! But of course, we only have point estimates of the parameters of our network.

Let's check convergence.

In [ ]:
# Plot losses
plt.figure(figsize=(6, 4)) 
plt.plot(map_losses)
plt.xlabel('Step')
plt.ylabel('Loss')
plt.title('MAP Training Loss')
plt.show()

The trace of loss does not show problematic signs.

Again, let's take samples from the posterior predictive and save some summaries.

In [ ]:
# Sample from the posterior predictive distribution
num_samples=2000
posterior_predictive = Predictive(model=model, guide=map_guide, params=map_params, num_samples=num_samples)
map_predictions = posterior_predictive(rng_key = rng_key_predict, X=X_test, Y=None, D_H=D_H, D_Y=1)
map_predictions_grid = posterior_predictive(rng_key = rng_key_predict, X=X_grid, Y=None, D_H=D_H, D_Y=1)

In [ ]:
# Extract the mean and quantiles for the predicted output on the test data
mean_test_map = jnp.mean(map_predictions['Y'], axis = 0).reshape(-1,)
lower_test_map = jnp.quantile(map_predictions['Y'], 0.025, axis = 0).reshape(-1,)
upper_test_map = jnp.quantile(map_predictions['Y'], 0.975, axis = 0).reshape(-1,)

# Extract the mean and quantiles for the predicted function on the grid data
mean_grid_map = jnp.mean(map_predictions_grid['z2'], axis = 0).reshape(-1,)
lower_grid_map = jnp.quantile(map_predictions_grid['z2'], 0.025, axis = 0).reshape(-1,)
upper_grid_map = jnp.quantile(map_predictions_grid['z2'], 0.975, axis = 0).reshape(-1,)

### Visualizing and comparing predictions

Now let's visualize and compare the predictions from the different inference algorithms.

In [ ]:
# Predictions with error bars
figs, axs = plt.subplots(1, 3, figsize=(18, 5))
# MCMC predictions
axs[0].plot(X_test[:, 0], y_test, "rx", label="True data")
axs[0].plot(X_train[:, 0], y_train, "k.", label="Train data")
axs[0].plot(X_grid[:, 0], mean_grid_mcmc, "b-", label="Posterior Predictive Mean")
axs[0].errorbar(
    X_test[:, 0],
    mean_test_mcmc,
    yerr=[mean_test_mcmc-lower_test_mcmc, upper_test_mcmc-mean_test_mcmc],
    fmt="o",
    capsize=3,
    label="Posterior Predictive"
)

axs[0].set_xlabel("Time")
axs[0].set_ylabel("Acceleration")
axs[0].set_title("Posterior Predictive (MCMC) with 95% Credible Interval")
axs[0].legend()
# VI predictions
axs[1].plot(X_test[:, 0], y_test, "rx", label="True data")
axs[1].plot(X_train[:, 0], y_train, "k.", label="Train data")
axs[1].plot(X_grid[:, 0], mean_grid_svi, "b-", label="Posterior Predictive Mean")
axs[1].errorbar(
    X_test[:, 0],
    mean_test_svi,
    yerr=[mean_test_svi-lower_test_svi, upper_test_svi-mean_test_svi],
    fmt="o",
    capsize=3,
    label="Posterior Predictive"
)
axs[1].set_xlabel("Time")
axs[1].set_ylabel("Acceleration")
axs[1].set_title("Posterior Predictive (VI) with 95% Credible Interval")
axs[1].legend()
# MAP predictions
axs[2].plot(X_test[:, 0], y_test, "rx", label="True data")
axs[2].plot(X_train[:, 0], y_train, "k.", label="Train data")
axs[2].plot(X_grid[:, 0], mean_grid_map, "b-", label="Posterior Predictive Mean")
axs[2].errorbar(
    X_test[:, 0],
    mean_test_map,
    yerr=[mean_test_map-lower_test_map, upper_test_map-mean_test_map],
    fmt="o",
    capsize=3,
    label="Posterior Predictive"
)
axs[2].set_xlabel("Time")
axs[2].set_ylabel("Acceleration")
axs[2].set_title("Posterior Predictive (MAP) with 95% Credible Interval")
axs[2].legend()
plt.tight_layout()
plt.show()

In [ ]:
# Predictions with error bars
figs, axs = plt.subplots(1, 3, figsize=(18, 5))
# MCMC predictions
axs[0].plot(X_train[:, 0], y_train, "k.", label="Train data")
axs[0].plot(X_grid[:, 0], mean_grid_mcmc, "b-", label="Posterior Predictive Mean")
axs[0].fill_between(
    X_grid[:, 0], lower_grid_mcmc, upper_grid_mcmc, alpha = 0.2, label = '95% Credible Interval')
axs[0].set_xlabel("Time")
axs[0].set_ylabel("Acceleration")
axs[0].set_title("Posterior Predictive (MCMC) for the function")
axs[0].legend()
# VI predictions
axs[1].plot(X_train[:, 0], y_train, "k.", label="Train data")
axs[1].plot(X_grid[:, 0], mean_grid_svi, "b-", label="Posterior Predictive Mean")
axs[1].fill_between(
    X_grid[:, 0], lower_grid_svi, upper_grid_svi, alpha = 0.2, label = '95% Credible Interval')
axs[1].set_xlabel("Time")
axs[1].set_ylabel("Acceleration")
axs[1].set_title("Posterior Predictive (VI) for the function")
axs[1].legend()
# MAP predictions
axs[2].plot(X_train[:, 0], y_train, "k.", label="Train data")
axs[2].plot(X_grid[:, 0], mean_grid_map, "b-", label="Posterior Predictive Mean")
axs[2].fill_between(
    X_grid[:, 0], lower_grid_map, upper_grid_map, alpha = 0.2, label = '95% Credible Interval')
axs[2].set_xlabel("Time")
axs[2].set_ylabel("Acceleration")
axs[2].set_title("Posterior Predictive (MAP) for the function")
axs[2].legend()
plt.tight_layout()
plt.show()

In [ ]:
# MSE and R2
from sklearn.metrics import mean_squared_error, r2_score

print('MCMC: MSE is ', mean_squared_error(y_test,mean_test_mcmc))
print('MCMC: R2 is ', r2_score(y_test,mean_test_mcmc))
print('VI: MSE is ', mean_squared_error(y_test,mean_test_svi))
print('VI: R2 is ', r2_score(y_test,mean_test_svi))
print('MAP: MSE is ', mean_squared_error(y_test,mean_test_map))
print('MAP: R2 is ', r2_score(y_test,mean_test_map))

Overall, the performance is quite similar. With MAP, we of course only have a point estimate of the function, and VI tends to produce slightly wider CIs for the function and observations. 

To quantify how well the CIs capture the uncertainty in predictions, we compare both the empirical coverage for the test points as well as the average width of the CIs. Ideally, we would like coverage close to the nominal level and narrow CIs! 

In [ ]:
# Compute empirical coverage and average width of credible intervals for all three methods (MCMC, VI, MAP)
q = np.arange(0.5,1,0.01)
ec_mcmc = np.zeros(q.shape)
ci_avg_width_mcmc = np.zeros(q.shape)
ec_vi = np.zeros(q.shape)
ci_avg_width_vi = np.zeros(q.shape)
ec_map = np.zeros(q.shape)
ci_avg_width_map = np.zeros(q.shape)
N_test = y_test.shape[0]

for i in range(q.shape[0]):
  # Compute credible interval (MCMC)
  lower_pred = np.quantile(predictions['Y'], q = (1-q[i])/2, axis = 0)
  upper_pred = np.quantile(predictions['Y'], q = q[i]+ (1-q[i])/2, axis = 0)
  # Summarize coverage
  ec_mcmc[i]= np.sum((y_test>lower_pred)&(y_test<upper_pred))/N_test
  # Summarize average width
  ci_avg_width_mcmc[i] = np.mean(upper_pred - lower_pred)
  # Compute credible interval (VI)
  lower_pred = np.quantile(svi_predictions['Y'], q = (1-q[i])/2, axis = 0)
  upper_pred = np.quantile(svi_predictions['Y'], q = q[i]+ (1-q[i])/2, axis = 0)
  # Summarize coverage
  ec_vi[i]= np.sum((y_test>lower_pred)&(y_test<upper_pred))/N_test
  # Summarize average width
  ci_avg_width_vi[i] = np.mean(upper_pred - lower_pred)
  # Compute credible interval (MAP)
  lower_pred = np.quantile(map_predictions['Y'], q = (1-q[i])/2, axis = 0)
  upper_pred = np.quantile(map_predictions['Y'], q = q[i]+ (1-q[i])/2, axis = 0)
  # Summarize coverage
  ec_map[i]= np.sum((y_test>lower_pred)&(y_test<upper_pred))/N_test
  # Summarize average width
  ci_avg_width_map[i] = np.mean(upper_pred - lower_pred)

# Plot empirical coverage and average width of credible intervals
fig, axs = plt.subplots(1, 2, figsize=(12, 5))
# Plot empirical coverage 
axs[0].plot(q, ec_vi, linewidth = 2, label = 'VI')
axs[0].plot(q, ec_mcmc, linewidth = 2, label = 'MCMC')
axs[0].plot(q, ec_map, linewidth = 2, label = 'MAP')
axs[0].plot(q,q, color='k',linestyle = 'dashed', linewidth = 2)
axs[0].legend()
axs[0].set_xlabel('Quantile')
axs[0].set_ylabel('Coverage')
axs[0].set_title('Empirical Coverage')
# Plot average width
axs[1].plot(q, ci_avg_width_vi, linewidth = 2, label = 'VI')
axs[1].plot(q, ci_avg_width_mcmc, linewidth = 2, label = 'MCMC')
axs[1].plot(q, ci_avg_width_map, linewidth = 2, label = 'MAP')
axs[1].legend()
axs[1].set_xlabel('Quantile')
axs[1].set_ylabel('Average Width')
axs[1].set_title('Credible Interval Width')
plt.tight_layout()
plt.show()

In general, we observe that MCMC provides the best balance between empirical coverage and CI width, but the speed-up is non-neglible, and for larger datasets and models, one may prefer VI, which has slightly wider CIs but similar empirical coverage (MAP tends to slightly under-cover).

# BNN: one layer, large width, odd activation <a id='bnn2'></a>

Is only 10 hidden units sufficiently flexible? Let's try increasing the width.

In [ ]:
# Render and visualize the model
D_H = 100
D_Y = 1
numpyro.render_model(model, model_args=(X_train, y_train, D_H, D_Y),
    render_distributions=True,
    render_params=True,)

## Inference

Again, we will compare the three different inference algorithms.

### MCMC

Let's start with MCMC. 

In [ ]:
# Run MCMC
num_warmup=2000
num_samples=5000
num_chains=1
numpyro.set_host_device_count(num_chains)

posterior_samples = run_mcmc(model = model,  num_chains = num_chains,num_samples = num_samples, num_warmup = num_warmup, rng_key = rng_key, 
                             X = X_train, Y = y_train, D_H = D_H, D_Y = D_Y)

That took a bit longer, as expected since the number of parameters has increased. **How many parameters are in the model?**

In [ ]:
# Traceplots
fig, axs = plt.subplots(1, 3, figsize=(12, 4))
axs = axs.flatten()
# Traceplot for tau
axs[0].plot(posterior_samples['tau'])
axs[0].set_title('Traceplot for tau')
axs[0].set_xlabel('iteration')
axs[0].set_ylabel('tau')
# Traceplot for z2
axs[1].plot(posterior_samples['z2'][:,0,0])
axs[1].set_title('Traceplot of the mean for x=' + str(np.round(X_train[0,0], 2)))
axs[1].set_xlabel('iteration')
axs[1].set_ylabel('mean')
# Traceplot for z2
axs[2].plot(posterior_samples['z2'][:,1,0])
axs[2].set_title('Traceplot of the mean for x=' + str(np.round(X_train[1,0], 2)))
axs[2].set_xlabel('iteration')
axs[2].set_ylabel('mean')
plt.show()

The traceplots look ok. We proceed to predictive sampling.

In [ ]:
# Draw predictive samples from the posterior predictive distribution
predictive = Predictive(model = model, posterior_samples=posterior_samples)
predictions = predictive(rng_key = rng_key_predict, X=X_test, Y = None, D_H = D_H, D_Y = D_Y)
predictions_grid = predictive(rng_key = rng_key_predict, X=X_grid, Y = None, D_H = D_H, D_Y = D_Y)

In [ ]:
# Extract the mean and quantiles for the predicted output on the test data
mean_test_mcmc = jnp.mean(predictions['Y'], axis = 0).reshape(-1,)
lower_test_mcmc = jnp.quantile(predictions['Y'], 0.025, axis = 0).reshape(-1,)
upper_test_mcmc = jnp.quantile(predictions['Y'], 0.975, axis = 0).reshape(-1,)

# Extract the mean and quantiles for the predicted function on the grid data
mean_grid_mcmc = jnp.mean(predictions_grid['z2'], axis = 0).reshape(-1,)
lower_grid_mcmc = jnp.quantile(predictions_grid['z2'], 0.025, axis = 0).reshape(-1,)
upper_grid_mcmc = jnp.quantile(predictions_grid['z2'], 0.975, axis = 0).reshape(-1,)

### SVI

Next, let's use SVI.

In [ ]:
# Run SVI
numsteps= 1000
svi_losses, svi_params, svi_guide = do_SVI(model=model, rng_key=rng_key, numsteps=numsteps, X = X_train, Y = y_train, D_H=D_H, D_Y=D_Y)

In [ ]:
# Plot the ELBO to check convergence
plt.figure(figsize=(6, 4))
sns.lineplot(x=np.arange(numsteps), y=-svi_losses)
plt.xlabel('Step')
plt.ylabel('ELBO')
plt.title('ELBO over steps')

Convergence looks ok. Let's proceed to predictive sampling.

In [ ]:
# Sample from the posterior predictive distribution
num_samples=2000
posterior_predictive = Predictive(model=model, guide=svi_guide, params=svi_params, num_samples=num_samples)
svi_predictions = posterior_predictive(rng_key = rng_key_predict, X=X_test, Y=None, D_H=D_H, D_Y=D_Y)
svi_grid = posterior_predictive(rng_key = rng_key_predict, X=X_grid, Y=None, D_H=D_H, D_Y=D_Y)

In [ ]:
# Extract the mean and quantiles for the predicted output on the test data
mean_test_svi = jnp.mean(svi_predictions['Y'], axis = 0).reshape(-1,)
lower_test_svi = jnp.quantile(svi_predictions['Y'], 0.025, axis = 0).reshape(-1,)
upper_test_svi = jnp.quantile(svi_predictions['Y'], 0.975, axis = 0).reshape(-1,)

# Extract the mean and quantiles for the predicted function on the grid data
mean_grid_svi = jnp.mean(svi_grid['z2'], axis = 0).reshape(-1,)
lower_grid_svi = jnp.quantile(svi_grid['z2'], 0.025, axis = 0).reshape(-1,)
upper_grid_svi = jnp.quantile(svi_grid['z2'], 0.975, axis = 0).reshape(-1,)

### MAP

Lastly, let's use MAP inference.

In [ ]:
# Run MAP estimation
numsteps= 1000
map_losses, map_params, map_guide = do_MAP(model=model, rng_key=rng_key, numsteps=numsteps, X = X_train, Y = y_train, D_H = D_H, D_Y = 1)

In [ ]:
# Plot losses
plt.figure(figsize=(6, 4)) 
plt.plot(map_losses)
plt.xlabel('Step')
plt.ylabel('Loss')
plt.title('MAP Training Loss')
plt.show()

Convergence looks ok. Let's proceed to predictive sampling.

In [ ]:
# Sample from the posterior predictive distribution
num_samples=2000
posterior_predictive = Predictive(model=model, guide=map_guide, params=map_params, num_samples=num_samples)
map_predictions = posterior_predictive(rng_key = rng_key_predict, X=X_test, Y=None, D_H=D_H, D_Y=1)
map_predictions_grid = posterior_predictive(rng_key = rng_key_predict, X=X_grid, Y=None, D_H=D_H, D_Y=1)

In [ ]:
# Extract the mean and quantiles for the predicted output on the test data
mean_test_map = jnp.mean(map_predictions['Y'], axis = 0).reshape(-1,)
lower_test_map = jnp.quantile(map_predictions['Y'], 0.025, axis = 0).reshape(-1,)
upper_test_map = jnp.quantile(map_predictions['Y'], 0.975, axis = 0).reshape(-1,)

# Extract the mean and quantiles for the predicted function on the grid data
mean_grid_map = jnp.mean(map_predictions_grid['z2'], axis = 0).reshape(-1,)
lower_grid_map = jnp.quantile(map_predictions_grid['z2'], 0.025, axis = 0).reshape(-1,)
upper_grid_map = jnp.quantile(map_predictions_grid['z2'], 0.975, axis = 0).reshape(-1,)

### Visualizing and comparing predictions

Again, let's compare the predictions.

In [ ]:
# Predictions with error bars
figs, axs = plt.subplots(1, 3, figsize=(18, 5))
# MCMC predictions
axs[0].plot(X_test[:, 0], y_test, "rx", label="True data")
axs[0].plot(X_train[:, 0], y_train, "k.", label="Train data")
axs[0].plot(X_grid[:, 0], mean_grid_mcmc, "b-", label="Posterior Predictive Mean")
axs[0].errorbar(
    X_test[:, 0],
    mean_test_mcmc,
    yerr=[mean_test_mcmc-lower_test_mcmc, upper_test_mcmc-mean_test_mcmc],
    fmt="o",
    capsize=3,
    label="Posterior Predictive"
)

axs[0].set_xlabel("Time")
axs[0].set_ylabel("Acceleration")
axs[0].set_title("Posterior Predictive (MCMC) with 95% Credible Interval")
axs[0].legend()
# VI predictions
axs[1].plot(X_test[:, 0], y_test, "rx", label="True data")
axs[1].plot(X_train[:, 0], y_train, "k.", label="Train data")
axs[1].plot(X_grid[:, 0], mean_grid_svi, "b-", label="Posterior Predictive Mean")
axs[1].errorbar(
    X_test[:, 0],
    mean_test_svi,
    yerr=[mean_test_svi-lower_test_svi, upper_test_svi-mean_test_svi],
    fmt="o",
    capsize=3,
    label="Posterior Predictive"
)
axs[1].set_xlabel("Time")
axs[1].set_ylabel("Acceleration")
axs[1].set_title("Posterior Predictive (VI) with 95% Credible Interval")
axs[1].legend()
# MAP predictions
axs[2].plot(X_test[:, 0], y_test, "rx", label="True data")
axs[2].plot(X_train[:, 0], y_train, "k.", label="Train data")
axs[2].plot(X_grid[:, 0], mean_grid_map, "b-", label="Posterior Predictive Mean")
axs[2].errorbar(
    X_test[:, 0],
    mean_test_map,
    yerr=[mean_test_map-lower_test_map, upper_test_map-mean_test_map],
    fmt="o",
    capsize=3,
    label="Posterior Predictive"
)
axs[2].set_xlabel("Time")
axs[2].set_ylabel("Acceleration")
axs[2].set_title("Posterior Predictive (MAP) with 95% Credible Interval")
axs[2].legend()
axs[0].legend()
plt.tight_layout()
plt.show()

In [ ]:
# Predictions with error bars
figs, axs = plt.subplots(1, 3, figsize=(18, 5))
# MCMC predictions
axs[0].plot(X_train[:, 0], y_train, "k.", label="Train data")
axs[0].plot(X_grid[:, 0], mean_grid_mcmc, "b-", label="Posterior Predictive Mean")
axs[0].fill_between(
    X_grid[:, 0], lower_grid_mcmc, upper_grid_mcmc, alpha = 0.2, label = '95% Credible Interval')
axs[0].set_xlabel("Time")
axs[0].set_ylabel("Acceleration")
axs[0].set_title("Posterior Predictive (MCMC) for the function")
axs[0].legend()
# VI predictions
axs[1].plot(X_train[:, 0], y_train, "k.", label="Train data")
axs[1].plot(X_grid[:, 0], mean_grid_svi, "b-", label="Posterior Predictive Mean")
axs[1].fill_between(
    X_grid[:, 0], lower_grid_svi, upper_grid_svi, alpha = 0.2, label = '95% Credible Interval')
axs[1].set_xlabel("Time")
axs[1].set_ylabel("Acceleration")
axs[1].set_title("Posterior Predictive (VI) for the function")
axs[1].legend()
# MAP predictions
axs[2].plot(X_train[:, 0], y_train, "k.", label="Train data")
axs[2].plot(X_grid[:, 0], mean_grid_map, "b-", label="Posterior Predictive Mean")
axs[2].fill_between(
    X_grid[:, 0], lower_grid_map, upper_grid_map, alpha = 0.2, label = '95% Credible Interval')
axs[2].set_xlabel("Time")
axs[2].set_ylabel("Acceleration")
axs[2].set_title("Posterior Predictive (MAP) for the function")
axs[2].legend()
axs[0].legend()
plt.tight_layout()
plt.show()

In [ ]:
# MSE and R2
from sklearn.metrics import mean_squared_error, r2_score

print('MCMC: MSE is ', mean_squared_error(y_test,mean_test_mcmc))
print('MCMC: R2 is ', r2_score(y_test,mean_test_mcmc))
print('VI: MSE is ', mean_squared_error(y_test,mean_test_svi))
print('VI: R2 is ', r2_score(y_test,mean_test_svi))
print('MAP: MSE is ', mean_squared_error(y_test,mean_test_map))
print('MAP: R2 is ', r2_score(y_test,mean_test_map))

In [ ]:
# Compute empirical coverage and average width of credible intervals for all three methods (MCMC, VI, MAP)
q = np.arange(0.5,1,0.01)
ec_mcmc = np.zeros(q.shape)
ci_avg_width_mcmc = np.zeros(q.shape)
ec_vi = np.zeros(q.shape)
ci_avg_width_vi = np.zeros(q.shape)
ec_map = np.zeros(q.shape)
ci_avg_width_map = np.zeros(q.shape)
N_test = y_test.shape[0]

for i in range(q.shape[0]):
  # Compute credible interval (MCMC)
  lower_pred = np.quantile(predictions['Y'], q = (1-q[i])/2, axis = 0)
  upper_pred = np.quantile(predictions['Y'], q = q[i]+ (1-q[i])/2, axis = 0)
  # Summarize coverage
  ec_mcmc[i]= np.sum((y_test>lower_pred)&(y_test<upper_pred))/N_test
  # Summarize average width
  ci_avg_width_mcmc[i] = np.mean(upper_pred - lower_pred)
  # Compute credible interval (VI)
  lower_pred = np.quantile(svi_predictions['Y'], q = (1-q[i])/2, axis = 0)
  upper_pred = np.quantile(svi_predictions['Y'], q = q[i]+ (1-q[i])/2, axis = 0)
  # Summarize coverage
  ec_vi[i]= np.sum((y_test>lower_pred)&(y_test<upper_pred))/N_test
  # Summarize average width
  ci_avg_width_vi[i] = np.mean(upper_pred - lower_pred)
  # Compute credible interval (MAP)
  lower_pred = np.quantile(map_predictions['Y'], q = (1-q[i])/2, axis = 0)
  upper_pred = np.quantile(map_predictions['Y'], q = q[i]+ (1-q[i])/2, axis = 0)
  # Summarize coverage
  ec_map[i]= np.sum((y_test>lower_pred)&(y_test<upper_pred))/N_test
  # Summarize average width
  ci_avg_width_map[i] = np.mean(upper_pred - lower_pred)

# Plot empirical coverage and average width of credible intervals
fig, axs = plt.subplots(1, 2, figsize=(12, 5))
# Plot empirical coverage 
axs[0].plot(q, ec_vi, linewidth = 2, label = 'VI')
axs[0].plot(q, ec_mcmc, linewidth = 2, label = 'MCMC')
axs[0].plot(q, ec_map, linewidth = 2, label = 'MAP')
axs[0].plot(q,q, color='k',linestyle = 'dashed', linewidth = 2)
axs[0].legend()
axs[0].set_xlabel('Quantile')
axs[0].set_ylabel('Coverage')
axs[0].set_title('Empirical Coverage')
# Plot average width
axs[1].plot(q, ci_avg_width_vi, linewidth = 2, label = 'VI')
axs[1].plot(q, ci_avg_width_mcmc, linewidth = 2, label = 'MCMC')
axs[1].plot(q, ci_avg_width_map, linewidth = 2, label = 'MAP')
axs[1].legend()
axs[1].set_xlabel('Quantile')
axs[1].set_ylabel('Average Width')
axs[1].set_title('Credible Interval Width')
plt.tight_layout()
plt.show()

What happened with VI??? The results now look terrible!!

This phenomenon was studied in [Cocker et al (2022)](https://proceedings.mlr.press/v151/coker22a.html), where they showed that for fully-connected BNNs with odd activation functions and a Gaussian likelihood:
- the **optimal mean-field variational posterior predictive converges to the prior predictive as the width tends to infinity,**
- instead, the **true posterior predictive converges to the NNGP (Gaussian process with neural network kernel).**

The authors provide non-asymptotic convergence bounds to characterize this phenomenon, however, they are too loose for practical guidance (convergence is often faster than suggested by the bounds).

Some further insights:
- They show that this behavior arises because the KL regularization is too strong; this provides insight into why *cold posteriors* that downweight the KL regularization can empirically yield better performance (this can alternatively be viewed as *power posteriors* that raise the likelihood to a power>1).
- Their results suggest that this hold for more general likelihood beyond the Gaussian homoskedastic likelihood.
- Their bound that the width needs to be larger for deep networks compared to shallow networks; however, these are loose bounds and convergence could be faster. In particular, the first two moments of the optimal variational posterior predictive converge to the prior predictive moments if 
$$ \lim_{N,D_H \rightarrow \infty}  \frac{N^{L+1}}{D_H} = 0, \text{ where } D_H \text{ is the common width of all } L \text{ hidden layers.}$$
- Thus, inaccurate (mean-field variational) inference combined with very large models are limiting in BNNs.

Some open questions:
- They study the case of Gaussian priors, does this extend to other choices of priors?
- Can issues be avoided by using non-odd activations, such as ReLU? Check [Thibault's google scholar](https://scholar.google.com/citations?user=329uecAAAAAJ&hl=en&oi=ao) for a forthcoming preprint...
- Can we still use mean-field VI and merely be careful about the relative scaling and choices of width, depth, and data size?
- Does this extend to more relaxed assumptions on the variational posterior?


# Exercises <a id='ex'></a>
1. Trying changing the width to identify when VI starts to break down.
2. Changing the activation to ReLU (`jnp.maximum(a, 0)`). Do you still observe the same behavior? 
3. Try changing the hyperparameters of the Gamma prior and/or multipling the noise variance `sigma_obs` by a constant (a constant smaller than 1 mimics the cold posterior in the case of a Gaussian likelihood - does this help to combat and/or alter the rate of convergence to the prior predictive for SVI?)
4. Try changing the prior, e.g. to Laplace (`dist.Laplace`). 
5. Try changing to the assumptions on the variational posterior, e.g. to `AutoMultivariateNormal` to remove the mean-field assumption.
6. Try adding changing the number of layers (see code below).

In [ ]:
# Define the model
# D_H: list of hidden layer sizes
def model(X, Y, D_H, D_Y=1):
    N, D_0 = X.shape
    x = deterministic("x", X) 

    L = len(D_H) if isinstance(D_H, (list, tuple)) else 1

    # Hidden layers of the neural network
    for l in range(L):
        D_in = D_0 if l == 0 else D_H[l-1]
        D_out = D_H[l] if isinstance(D_H, (list, tuple)) else D_H

        # Bias term for the current layer
        b = numpyro.sample(f"b{l+1}", dist.Normal(jnp.zeros((D_out,)), jnp.ones((D_out,))))
        assert b.shape == (D_out,)
        # Weights for the current layer
        w = numpyro.sample(f"w{l+1}", dist.Normal(jnp.zeros((D_in, D_out)), jnp.ones((D_in, D_out))/ jnp.sqrt(D_in)))
        assert w.shape == (D_in, D_out)
        # Activations for the current layer
        x = deterministic(f"z{l+1}", acti_fun(x @ w + b)) 
        assert x.shape == (N, D_out)

    # Final layer of the neural network
    # Bias term for the final layer
    b2 = numpyro.sample("b_final", dist.Normal(jnp.zeros((D_Y,)), jnp.ones((D_Y,))))
    assert b2.shape == (D_Y,)
    # Weights for the second layer
    D_in = D_H if L == 1 else D_H[L-1]
    w2 = numpyro.sample("w_final", dist.Normal(jnp.zeros((D_in, D_Y)), jnp.ones((D_in, D_Y))/ jnp.sqrt(D_in)))
    assert w2.shape == (D_in, D_Y)
    # Activations for the second layer
    z2 = deterministic("z_final", x @ w2 + b2)
    assert z2.shape == (N, D_Y)

    if Y is not None:
        assert z2.shape == Y.shape

    # we put a prior on the observation noise
    prec_obs = numpyro.sample("tau", dist.Gamma(100.0, 1.0))
    sigma_obs = 1.0 / jnp.sqrt(prec_obs)

    # observe data
    with numpyro.plate("N", N):
        # note we use to_event(1) because each observation has shape (1,)
        numpyro.sample("Y", dist.Normal(z2, sigma_obs).to_event(1), obs=Y)

In [ ]:
# Render and visualize the model
D_H = [10,10,10]
D_Y = 1
numpyro.render_model(model, model_args=(X_train, y_train, D_H, D_Y),
    render_distributions=True,
    render_params=True,)

In [ ]:
# Run SVI
numsteps= 1000
svi_losses, svi_params, svi_guide = do_SVI(model=model, rng_key=rng_key, numsteps=numsteps, X = X_train, Y = y_train, D_H=D_H, D_Y=D_Y)

In [ ]:
# Plot the ELBO to check convergence
plt.figure(figsize=(6, 4))
sns.lineplot(x=np.arange(numsteps), y=-svi_losses)
plt.xlabel('Step')
plt.ylabel('ELBO')
plt.title('ELBO over steps')

In [ ]:
# Sample from the posterior predictive distribution
num_samples=2000
posterior_predictive = Predictive(model=model, guide=svi_guide, params=svi_params, num_samples=num_samples)
svi_predictions = posterior_predictive(rng_key = rng_key_predict, X=X_test, Y=None, D_H=D_H, D_Y=D_Y)
svi_grid = posterior_predictive(rng_key = rng_key_predict, X=X_grid, Y=None, D_H=D_H, D_Y=D_Y)

In [ ]:
# Extract the mean and quantiles for the predicted output on the test data
mean_test_svi = jnp.mean(svi_predictions['Y'], axis = 0).reshape(-1,)
lower_test_svi = jnp.quantile(svi_predictions['Y'], 0.025, axis = 0).reshape(-1,)
upper_test_svi = jnp.quantile(svi_predictions['Y'], 0.975, axis = 0).reshape(-1,)

# Extract the mean and quantiles for the predicted function on the grid data
mean_grid_svi = jnp.mean(svi_grid['z_final'], axis = 0).reshape(-1,)
lower_grid_svi = jnp.quantile(svi_grid['z_final'], 0.025, axis = 0).reshape(-1,)
upper_grid_svi = jnp.quantile(svi_grid['z_final'], 0.975, axis = 0).reshape(-1,)

In [ ]:
# Predictions with error bars
figs, axs = plt.subplots(1, 1, figsize=(5, 5))
# VI predictions
axs.plot(X_train[:, 0], y_train, "k.", label="Train data")
axs.plot(X_grid[:, 0], mean_grid_svi, "b-", label="Posterior Predictive Mean")
axs.fill_between(
    X_grid[:, 0], lower_grid_svi, upper_grid_svi, alpha = 0.2, label = '95% Credible Interval')
axs.set_xlabel("Time")
axs.set_ylabel("Acceleration")
axs.set_title("Posterior Predictive (VI) for the function")
axs.legend()
plt.tight_layout()
plt.show()